In [1]:
import urllib.request
import pandas as pd

# Notebook contents 

This notebook is where I tested how to scrape the USNO website to pull data for the sun's position vs time for any given day. The URL can be edited to specify the date, location on earth, and time zone. The webpage data is then parsed and exported to a csv file titled `YYYY_DDD_CHIME.csv` with columns: `time`, `altitude`, `azimuth`, `date`, `latitude_degrees`, `longitude_degrees`, `time_zone`

In [2]:
url = "https://aa.usno.navy.mil/calculated/altaz?body=10&date=2024-05-15&intv_mag=1&lat=38.4353&lon=-79.8261&label=CHIME_15-may-2024&tz=4.00&tz_sign=-1&submit=Get+Data"
page = urllib.request.urlopen(url)

In [3]:
with open("/users/dbautist/CHIME_landing_directory/sunPosition/outfile.txt", "w") as f:
    data = page.read().decode("utf-8")
    f.write(data)

In [4]:
times = []
altitude = []
azimuth = []
first_row = data.split("</tr>")[0]
second_row = data.split("</tr>")[1]

In [5]:
def get_alt_az(data):
    times = []
    altitude = []
    azimuth = []
    rows = data.split("</tr>")[3:-1]
    for i in range(len(rows)):
        one_row = rows[i].split()[1:]
        if "dropped" in one_row:
            break
        altitude.append(float(one_row[1].replace("<td>", "").replace("</td>", "")[:-1]))
        azimuth.append(float(one_row[2].replace("<td>", "").replace("</td>", "")[:-1]))
        times.append(one_row[0].replace("<td>", "").replace("</td>", ""))

    data_dict = {"time":times, "altitude":altitude, "azimuth":azimuth}
    df = pd.DataFrame(data_dict)

    return df


In [6]:
print(first_row)

<!doctype html>
<html lang="en">

<head>
    <meta charset="utf-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
   
    <title>Altitude and Azimuth of the Sun or Moon During One Day</title>


    <link rel="stylesheet" href="/uswds/css/uswds.css">
    <link rel="stylesheet" href="/css/styles.css">
    <link rel="stylesheet" href="/css/aa.css">
    <link rel="stylesheet" href="/css/pretty_out.css">
    <link rel="shortcut icon" href="/img/favicon.ico">
</head>

<body>
    

<a class="usa-skipnav" href="#main-content">Skip to main content</a>


<div class="site-banner usa-banner">
    <div class="usa-accordion">
        <header class="usa-banner__header">
            <div class="usa-banner__inner">
                <div class="grid-col-auto">
                    <img class="usa-banner__header-flag" src="/uswds/img/us_flag_small.png" alt="U.S. flag">
                </div>
                <div class="g

In [7]:
description = first_row.split("<th")[1]
title_contents = description.split("<br> ")
filename = title_contents[1].replace("\n","")
date = title_contents[3].replace("\n", "")
lat_dms, lon_dms = title_contents[2].split(";")[:2]
lon_dms = lon_dms.replace(", ", "")

In [8]:
def dms_to_deg(dms):
    if dms[0] == "N" or dms[0] == "E":
        direction = 1
    elif dms[0] == "S" or dms[0] == "W":
        direction = -1
    else:
        raise ValueError("I don't know what direction the coordinates are in. They should be N,S,E,W")
    degree, minute = dms.split()[1:3]
    degree = int(degree[:-1])
    minute = int(minute[:-4]) / 60
    return direction * (degree + minute)

In [9]:
def get_title_info(data):
    first_row = data.split("</tr>")[0]
    description = first_row.split("<th")[1]
    title_contents = description.split("<br> ")
    filename = title_contents[1].replace("\n","").replace(" ", "")
    date = title_contents[3].replace("\n", "").replace(" " , "")
    lat_dms, lon_dms = title_contents[2].split(";")[:2]
    lon_dms = lon_dms.replace(", ", "")

    lat_degrees = dms_to_deg(lat_dms)
    lon_degrees = dms_to_deg(lon_dms)

    return filename, date, lat_degrees, lon_degrees

In [10]:
get_title_info(data)

('CHIME_15-may-2024', '2024-05-15', 38.43333333333333, -79.83333333333333)

In [11]:
def query_website(day, month, year, latitude=38.43, longitude=-79.83, time_zone=-4, filename="default", ):
    if filename == "default":
        filename = f"CHIME_{day}-{month}-{year}"
    if time_zone > 0:
        time_zone_sign = 1
    elif time_zone < 0:
        time_zone = -1*time_zone
        time_zone_sign = -1
    else:
        time_zone = 0
        time_zone_sign = 1
    url_str = f"https://aa.usno.navy.mil/calculated/altaz?body=10&date={year}-{month}-{day}&intv_mag=1&lat={latitude}&lon={longitude}&label={filename}&tz={time_zone}&tz_sign={time_zone_sign}&submit=Get+Data"
    print(f"querying: {url_str}")

    page = urllib.request.urlopen(url_str)
    data = page.read().decode("utf-8")

    return data

In [12]:
demo = query_website(12, 12, 2022, time_zone=8)

querying: https://aa.usno.navy.mil/calculated/altaz?body=10&date=2022-12-12&intv_mag=1&lat=38.43&lon=-79.83&label=CHIME_12-12-2022&tz=8&tz_sign=1&submit=Get+Data


In [13]:
get_title_info(demo)

('CHIME_12-12-2022', '2022-12-12', 38.43333333333333, -79.83333333333333)

In [14]:
get_alt_az(demo)

,time,altitude,azimuth
0,00:01,26.3,161.6
1,00:02,26.4,161.9
2,00:03,26.4,162.1
3,00:04,26.5,162.4
4,00:05,26.6,162.6
...,...,...,...
416,06:57,-11.2,249.4
417,06:58,-11.4,249.5
418,06:59,-11.6,249.6
419,07:00,-11.7,249.8
